# i-PROGNOSIS typing experiment

Run cells from top to bottom. The notebook exposes dataset/audit inspection and output paths; reusable parsing, feature extraction, and training logic remain in `src/iprognosis_data.py` and `src/train_typing_models.py`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').is_dir(), 'Open this notebook from the repository or notebooks directory'
sys.path.insert(0, str(PROJECT_ROOT))

from src.iprognosis_data import DEFAULT_IPROGNOSIS_ROOT, build_participant_dataset
from src.train_typing_models import run_experiment

DATA_ROOT = DEFAULT_IPROGNOSIS_ROOT
MODEL_PATH = PROJECT_ROOT / 'models' / 'finalization_baseline' / 'typing_iprognosis_model.joblib'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'finalization_baseline' / 'typing'
SEED = 42
print('Typing data root:', DATA_ROOT)
print('Data directory exists:', (DATA_ROOT / 'Data').is_dir())
print('Clinical CSV exists:', (DATA_ROOT / 'Demographics_Clinical_Characteristics.csv').is_file())

Typing data root: C:\Users\daisy\Projects\AI\pd-ml-lab\data\virtual keyboard data\iprognosis_typing_dev
Data directory exists: True
Clinical CSV exists: True


In [2]:
# Parse/link the local cohort and inspect participant labels and data-quality audit before training.
participants, excerpt_features, data_audit = build_participant_dataset(DATA_ROOT)
display(participants.groupby('target').size().rename('participants').to_frame())
display(participants.head())
display(excerpt_features.head())
data_audit

,participants
target,
0,15
1,18


,participant_id,target,hold_ms_mean,hold_ms_std,hold_ms_median,hold_ms_q90,hold_ms_iqr,hold_ms_cv,press_interval_ms_mean,press_interval_ms_std,...,pause_ms_cv,pressure_normalized_mean,pressure_normalized_std,pressure_normalized_median,pressure_normalized_q90,pressure_normalized_iqr,pressure_normalized_cv,pause_fraction,excerpt_count,total_valid_events
0,01,1,80.690909,17.538789,80.0,104.8,24.000,0.219043,1120.269231,808.689094,...,0.566254,0.535526,0.100252,0.57500,0.637500,0.175000,0.187641,0.335052,11,782
1,02,1,117.818182,23.108837,116.0,145.6,28.500,0.203190,1750.590909,1390.792010,...,0.555362,0.504779,0.116450,0.53125,0.612500,0.175000,0.237649,0.539216,11,1085
2,03,1,101.079208,31.278153,99.0,129.0,29.500,0.302652,1843.303571,1936.913409,...,0.809019,0.519100,0.110223,0.52500,0.662500,0.175000,0.218368,0.580645,11,963
3,04,1,161.150685,25.780965,160.0,188.8,31.000,0.172436,783.200000,548.801836,...,0.464617,0.579018,0.120918,0.55625,0.712500,0.175000,0.208833,0.119048,11,993
4,05,1,78.288462,22.070318,76.0,103.7,22.875,0.274266,864.119281,549.376261,...,0.403784,0.448996,0.106046,0.43750,0.595625,0.167188,0.233580,0.188948,10,756


,participant_id,excerpt_id,target,event_count_audit_only,hold_ms_mean,hold_ms_std,hold_ms_median,hold_ms_q90,hold_ms_iqr,hold_ms_cv,...,pause_ms_q90,pause_ms_iqr,pause_ms_cv,pressure_normalized_mean,pressure_normalized_std,pressure_normalized_median,pressure_normalized_q90,pressure_normalized_iqr,pressure_normalized_cv,pause_fraction
0,01,1,1,195,86.235897,20.543780,86.0,113.0,27.0,0.238228,...,5195.6,1545.00,0.864362,0.519872,0.104786,0.5625,0.6250,0.1750,0.201561,0.335052
1,01,2,1,49,85.224490,17.182413,83.0,107.2,24.0,0.201614,...,2240.1,600.00,0.324916,0.495663,0.098118,0.5000,0.6250,0.1750,0.197954,0.375000
2,01,3,1,57,75.614035,16.831210,74.0,93.6,18.0,0.222594,...,2682.8,926.00,0.994881,0.535526,0.100486,0.5750,0.6250,0.1375,0.187641,0.339286
3,01,4,1,47,90.510638,16.439279,90.0,109.8,19.0,0.181628,...,3360.0,706.00,0.566254,0.541223,0.090136,0.5750,0.6425,0.1750,0.166541,0.326087
4,01,5,1,57,80.070175,17.538789,81.0,98.4,19.0,0.219043,...,3745.1,1482.25,0.723219,0.538377,0.118281,0.5750,0.6500,0.1750,0.219700,0.357143


{'dataset': 'i-PROGNOSIS touchscreen typing development dataset',
 'label_source': 'Group column in Demographics_Clinical_Characteristics.csv; Control=0, PD=1',
 'participant_id_source': 'S## folder and S##_TEX## filename linked to Subject ID column',
 'event_schema': ['Press',
  'press timestamp milliseconds',
  'Release timestamp milliseconds',
  'normalized pressure 0-1'],
 'timestamp_unit': 'milliseconds, as documented in local ReadMe.txt',
 'pressure_summary_policy': 'Only observed values in documented [0,1] range enter pressure features; all values are retained in audit counts, no clipping',
 'aggregation': 'median of excerpt-level feature summaries per participant; event and excerpt counts are audit-only',
 'participants': 33,
 'participants_by_label': {'control': 15, 'pd': 18},
 'excerpt_files': 338,
 'excerpt_files_by_participant_min': 5,
 'excerpt_files_by_participant_max': 11,
 'audit_totals': {'nonblank_lines': 26343,
  'valid_events': 26343,
  'malformed_lines': 0,
  'inva

In [3]:
# Run participant holdout, training-only grouped CV, selection, test evaluation, and artifact writing.
summary = run_experiment(
    iprognosis_root=DATA_ROOT,
    model_path=MODEL_PATH,
    results_dir=RESULTS_DIR,
    seed=SEED,
)
summary

{
  "participants": 33,
  "participants_by_label": {
    "control": 15,
    "pd": 18
  },
  "excerpt_files": 338,
  "audit_totals": {
    "nonblank_lines": 26343,
    "valid_events": 26343,
    "malformed_lines": 0,
    "invalid_events": 0,
    "out_of_range_pressure_events": 7,
    "timestamp_order_reversals": 1,
    "negative_interkey_gaps_excluded": 1
  },
  "train_participants": 26,
  "test_participants": 7,
  "selected_model": "random_forest",
  "selected_hyperparameters": {
    "classifier__max_features": "sqrt",
    "classifier__min_samples_leaf": 2
  },
  "test_metrics": {
    "n_participants": 7,
    "class_support": {
      "control": 3,
      "pd": 4
    },
    "accuracy": 0.8571428571428571,
    "balanced_accuracy": 0.875,
    "precision_pd": 1.0,
    "recall_pd": 0.75,
    "f1_pd": 0.8571428571428571,
    "roc_auc": 1.0,
    "confusion_matrix_labels_control_pd": [
      [
        3,
        0
      ],
      [
        1,
        3
      ]
    ],
    "classification_report":

{'study_scope': 'Retrospective classification of known PD versus control status from smartphone touchscreen typing; not prospective risk prediction',
 'dataset_audit': {'dataset': 'i-PROGNOSIS touchscreen typing development dataset',
  'label_source': 'Group column in Demographics_Clinical_Characteristics.csv; Control=0, PD=1',
  'participant_id_source': 'S## folder and S##_TEX## filename linked to Subject ID column',
  'event_schema': ['Press',
   'press timestamp milliseconds',
   'Release timestamp milliseconds',
   'normalized pressure 0-1'],
  'timestamp_unit': 'milliseconds, as documented in local ReadMe.txt',
  'pressure_summary_policy': 'Only observed values in documented [0,1] range enter pressure features; all values are retained in audit counts, no clipping',
  'aggregation': 'median of excerpt-level feature summaries per participant; event and excerpt counts are audit-only',
  'participants': 33,
  'participants_by_label': {'control': 15, 'pd': 18},
  'excerpt_files': 338,


In [4]:
print('Model:', MODEL_PATH, MODEL_PATH.exists())
print('Results:', sorted(p.name for p in RESULTS_DIR.iterdir()))

Model: c:\Users\daisy\Projects\AI\pd-ml-lab\models\finalization_baseline\typing_iprognosis_model.joblib True
Results: ['typing_cv_results.csv', 'typing_excerpt_feature_audit.csv', 'typing_experiment_report.md', 'typing_participant_features.csv', 'typing_split_assignments.csv', 'typing_test_predictions.csv', 'typing_training_config.json', 'typing_training_metrics.json']
